# Exercises: Diving Deep into Vector Databases and RAG Chatbots
## Week 8 — Day 2 | DI GenAI & Machine Learning Bootcamp 2026

**What you'll build:**
- A functional RAG pipeline: text vectorization → FAISS index → ChromaDB → Hugging Face Q&A

**Topics:**
- Sentence Transformers embeddings
- FAISS (IndexFlatIP + IndexIDMap) cosine similarity search
- ChromaDB vector collection
- GPT-2 text generation for Q&A

## Exercise 1 — Data Loading and Preparation

In [ ]:
# Install required libraries
%pip install -q faiss-cpu numpy pandas sentence-transformers transformers torch
%pip install -q chromadb
print("Libraries installed ✓")

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import faiss
import chromadb
from sentence_transformers import SentenceTransformer, InputExample
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

warnings.filterwarnings("ignore")
os.makedirs("cache", exist_ok=True)

print(f"numpy   {np.__version__}")
print(f"faiss   {faiss.__version__}")
print(f"chromadb {chromadb.__version__}")
print("All imports OK ✓")

In [ ]:
# Download the labelled newscatcher dataset
import urllib.request

DATASET_URL = (
    "https://raw.githubusercontent.com/kotartemiy/topic-labeled-news-dataset"
    "/master/labelled_newscatcher_dataset.csv"
)
DATASET_PATH = "cache/labelled_newscatcher_dataset.csv"

if not os.path.exists(DATASET_PATH):
    print("Downloading dataset…")
    try:
        urllib.request.urlretrieve(DATASET_URL, DATASET_PATH)
        print(f"Downloaded → {DATASET_PATH}")
    except Exception as e:
        print(f"Download failed ({e}). Creating synthetic dataset for demo…")
        # Synthetic fallback so every cell can still run
        topics = ["SCIENCE", "TECHNOLOGY", "SPORTS", "WORLD", "BUSINESS",
                  "ENTERTAINMENT", "HEALTH", "POLITICS"]
        rng = np.random.default_rng(42)
        n = 2000
        sample_titles = [
            "NASA discovers new exoplanet in habitable zone",
            "Apple launches new MacBook with M3 chip",
            "World Cup final draws record viewership",
            "Stock markets rally on positive jobs data",
            "Scientists find evidence of ancient ocean on Mars",
            "Electric vehicle sales hit all-time high",
            "New study links sleep quality to heart health",
            "Summit on climate change opens in Dubai",
            "SpaceX successfully lands rocket for 100th time",
            "AI model beats human experts at protein folding",
        ] * (n // 10 + 1)
        df_syn = pd.DataFrame({
            "title": sample_titles[:n],
            "topic": [topics[i % len(topics)] for i in range(n)],
            "link": [f"https://example.com/article/{i}" for i in range(n)],
        })
        df_syn.to_csv(DATASET_PATH, sep=";", index=False)
        print(f"Synthetic dataset saved → {DATASET_PATH}")
else:
    print(f"Dataset already present → {DATASET_PATH}")

In [ ]:
# Load the dataset
path = DATASET_PATH

# Newscatcher CSV uses semicolon separator
try:
    pdf = pd.read_csv(path, sep=";")
except Exception:
    pdf = pd.read_csv(path)

print(f"Dataset shape   : {pdf.shape}")
print(f"Columns         : {list(pdf.columns)}")
print(f"\nTopic distribution:")
print(pdf["topic"].value_counts().to_string())

In [ ]:
# Add a unique integer ID column
pdf["id"] = range(len(pdf))

# Inspect
from IPython.display import display
display(pdf.head(10))

In [ ]:
# Create a subset of the first 1000 rows for faster processing
pdf_subset = pdf.head(1000).copy().reset_index(drop=True)

print(f"Full dataset  : {len(pdf):,} rows")
print(f"Subset        : {len(pdf_subset):,} rows")
display(pdf_subset.head())

## Exercise 2 — Vectorization with Sentence Transformers

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample

def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper: converts a news title string into an InputExample.
    guid  = positional index (auto-assigned by pandas apply)
    texts = [title]
    label = 0 (unused — we only need the text for embedding)
    """
    return InputExample(guid=None, texts=[str(doc1)], label=0)


faiss_train_examples = (
    pdf_subset
    .apply(lambda x: example_create_fn(x["title"]), axis=1)
    .tolist()
)

print(f"InputExample objects created : {len(faiss_train_examples)}")
print("\nFirst 10 examples:")
for ex in faiss_train_examples[:10]:
    print(f"  texts={ex.texts}")

In [ ]:
# Initialize the SentenceTransformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Model loaded: all-MiniLM-L6-v2")
print(f"Embedding dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
# Extract titles as a plain list of strings
titles_list = pdf_subset["title"].tolist()

print(f"Titles to embed: {len(titles_list)}")
print(f"Sample: {titles_list[:3]}")

In [ ]:
# Generate embeddings for all titles
print("Generating embeddings (this may take ~30s)…")
faiss_title_embedding = model.encode(
    titles_list,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)

print(f"\nEmbeddings generated ✓")
print(f"Number of embeddings : {len(faiss_title_embedding)}")
print(f"Embedding dimension  : {len(faiss_title_embedding[0])}")

In [ ]:
# Verify shape
n_embeddings, embedding_dim = len(faiss_title_embedding), len(faiss_title_embedding[0])
print(f"Shape check: ({n_embeddings}, {embedding_dim})")
print(f"  → {n_embeddings} titles each represented as a {embedding_dim}-dimensional vector")

## Exercise 3 — FAISS Indexing and Search

In [ ]:
import numpy as np
import faiss

# DataFrame to index and corresponding ID array
pdf_to_index = pdf_subset.copy()
id_index     = np.array(pdf_to_index["id"].tolist(), dtype=np.int64)

print(f"Articles to index : {len(pdf_to_index)}")
print(f"ID array shape    : {id_index.shape}")

In [ ]:
# Normalize embeddings to unit length → enables cosine similarity via inner product
content_encoded_normalized = faiss_title_embedding.copy().astype(np.float32)
faiss.normalize_L2(content_encoded_normalized)

# Verify: all norms should be ≈ 1.0 after normalization
norms = np.linalg.norm(content_encoded_normalized, axis=1)
print(f"Norm stats — min: {norms.min():.4f}  max: {norms.max():.4f}  mean: {norms.mean():.4f}")
print("Normalization ✓ (all norms ≈ 1.0)")

In [ ]:
# Build the FAISS index
# IndexFlatIP = exact inner product (= cosine similarity for normalized vectors)
# IndexIDMap  = maps search results back to original article IDs
embedding_dim = len(faiss_title_embedding[0])

index_content = faiss.IndexIDMap(faiss.IndexFlatIP(embedding_dim))
index_content.add_with_ids(content_encoded_normalized, id_index)

print(f"FAISS index built ✓")
print(f"  Index type      : IndexIDMap(IndexFlatIP)")
print(f"  Dimension       : {embedding_dim}")
print(f"  Vectors indexed : {index_content.ntotal}")

In [ ]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    """
    Encode the query, search the FAISS index, return the top-k matching articles.

    Parameters
    ----------
    query        : free-text query string
    pdf_to_index : DataFrame of indexed articles
    k            : number of nearest neighbours to retrieve

    Returns
    -------
    DataFrame with matched rows + a 'similarities' column.
    """
    # Encode the query string into a 1×D embedding vector
    query_vector = model.encode([query], convert_to_numpy=True).astype(np.float32)
    faiss.normalize_L2(query_vector)   # normalize → cosine similarity

    # Search: returns (similarities, ids) each of shape (1, k)
    top_k = index_content.search(query_vector, k)
    similarities = top_k[0][0]       # shape (k,)
    ids          = top_k[1][0]       # shape (k,) — original article IDs

    # Retrieve matching rows from the DataFrame
    results = pdf_to_index[pdf_to_index["id"].isin(ids)].copy()
    # Attach cosine similarity score to each result
    id_to_sim = dict(zip(ids.tolist(), similarities.tolist()))
    results["similarities"] = results["id"].map(id_to_sim)
    results = results.sort_values("similarities", ascending=False).reset_index(drop=True)
    return results


print("search_content() defined ✓")

In [ ]:
# Test the search function
print("=== Query: 'animal' — top 5 results ===")
display(search_content("animal", pdf_to_index, k=5))

print("\n=== Query: 'space exploration' — top 5 results ===")
display(search_content("space exploration", pdf_to_index, k=5))

print("\n=== Query: 'stock market crash' — top 5 results ===")
display(search_content("stock market crash", pdf_to_index, k=5))

## Exercise 4 — ChromaDB Collection and Querying

In [ ]:
import chromadb

# Initialize an ephemeral (in-memory) ChromaDB client
try:
    chroma_client = chromadb.EphemeralClient()   # chromadb >= 0.4
except AttributeError:
    chroma_client = chromadb.Client()            # chromadb 0.3.x fallback

collection_name = "my_news"

# Delete existing collection to avoid conflicts on re-runs
existing = [c.name for c in chroma_client.list_collections()]
if collection_name in existing:
    chroma_client.delete_collection(name=collection_name)
    print(f"Deleted existing collection '{collection_name}'")

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)
print(f"Collection created ✓  (ChromaDB v{chromadb.__version__})")

In [ ]:
# Inspect the subset before adding
display(pdf_subset.head())

# Add the first 100 titles to the ChromaDB collection
# ChromaDB auto-generates embeddings using its default SentenceTransformer
print("Adding 100 documents to the collection (auto-embedding, may take ~30s)…")

collection.add(
    documents = pdf_subset["title"][:100].tolist(),
    metadatas = [{"topic": str(t)} for t in pdf_subset["topic"][:100].tolist()],
    ids       = [str(i) for i in pdf_subset["id"][:100].tolist()],
)

print(f"\nDocuments in collection : {collection.count()}")

In [ ]:
# Query the collection for the top 10 most relevant documents
results = collection.query(
    query_texts = ["space"],
    n_results   = 10,
)

print("=== ChromaDB query: 'space' — top 10 results ===\n")
print(json.dumps(results, indent=4))

In [ ]:
# Pretty-print the retrieved documents with their topics and distances
print("=== Retrieved documents ===\n")
docs       = results["documents"][0]
metas      = results["metadatas"][0]
distances  = results["distances"][0] if results.get("distances") else ["N/A"] * len(docs)

for rank, (doc, meta, dist) in enumerate(zip(docs, metas, distances), 1):
    print(f"Rank {rank:>2} | topic={meta['topic']:<15} | dist={dist if dist=='N/A' else f'{dist:.4f}'}")
    print(f"         {doc}")
    print()

## Exercise 5 — Question Answering with a Hugging Face Model (RAG)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# We use GPT-2 — small, fast, no auth token required
model_id = "gpt2"

print(f"Loading tokenizer for '{model_id}'…")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"Loading causal LM '{model_id}'…")
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

print("Model loaded ✓")
n_params = sum(p.numel() for p in lm_model.parameters())
print(f"  Parameters : {n_params:,}")

In [ ]:
# Create the text-generation pipeline
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

pipe = pipeline(
    "text-generation",
    model          = lm_model,
    tokenizer      = tokenizer,
    max_new_tokens = 150,
    device_map     = "auto",
    pad_token_id   = tokenizer.eos_token_id,
)

print("Text-generation pipeline ready ✓")

In [ ]:
# Retrieve context from ChromaDB and build the RAG prompt
question = "What's the latest news on space development?"

# Query ChromaDB for the most relevant documents
chroma_results = collection.query(
    query_texts = [question],
    n_results   = 5,
)

context = " ".join(
    [f"#{i+1}: {doc}" for i, doc in enumerate(chroma_results["documents"][0])]
)

prompt_template = (
    f"Relevant context from recent news articles:\n{context}\n\n"
    f"Based on the context above, answer the following question:\n"
    f"Question: {question}\n"
    f"Answer:"
)

print("=== RAG Prompt ===")
print(prompt_template)
print("\n" + "=" * 60)

In [ ]:
# Generate a response using the pipeline
print("Generating response…\n")
lm_response = pipe(
    prompt_template,
    do_sample   = True,
    temperature = 0.7,
    top_k       = 50,
    top_p       = 0.95,
)

generated_text = lm_response[0]["generated_text"]
print("=== Generated Response ===")
print(generated_text)

In [ ]:
# Experiment with different questions and context sizes
experiments = [
    ("What are the major developments in artificial intelligence?", 5),
    ("Tell me about recent sports events.", 3),
    ("What is happening in global politics?", 4),
]

for question, n_ctx in experiments:
    print(f"\n{'='*65}")
    print(f"Question (context={n_ctx}): {question}")
    print("=" * 65)

    ctx_docs = collection.query(query_texts=[question], n_results=n_ctx)["documents"][0]
    context  = " ".join([f"#{i+1}: {d}" for i, d in enumerate(ctx_docs)])

    prompt = (
        f"Relevant news context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )

    resp = pipe(prompt, do_sample=True, temperature=0.7, top_k=50, top_p=0.95)
    # Print only the generated answer (after the prompt)
    answer = resp[0]["generated_text"][len(prompt):].strip()
    print(f"Answer: {answer[:300]}")

## Summary & Key Takeaways

| Step | Component | What it does |
|---|---|---|
| 1 | `pandas` | Load & inspect the labelled newscatcher dataset |
| 2 | `SentenceTransformer(all-MiniLM-L6-v2)` | Encode 1000 titles → 384-dim dense vectors |
| 3 | `faiss.IndexIDMap(IndexFlatIP)` | Exact cosine similarity search over normalized embeddings |
| 4 | `chromadb.EphemeralClient()` | Store 100 titles, auto-embed, query by semantic similarity |
| 5 | `gpt2` via `pipeline("text-generation")` | Generate answers conditioned on retrieved context (RAG) |

### FAISS vs ChromaDB

| Aspect | FAISS | ChromaDB |
|---|---|---|
| **Type** | Library | Database |
| **Embeddings** | You generate them | Auto-embedded or BYO |
| **Metadata** | Not supported natively | Built-in metadata storage |
| **Persistence** | Manual (write/load index) | Built-in (sqlite/parquet) |
| **Scale** | Billions of vectors | Millions of vectors |
| **Best for** | Speed-critical exact search | App development, RAG |

### RAG Pipeline Flow

```
User Question
     ↓
Embed query (SentenceTransformer)
     ↓
Vector similarity search (FAISS / ChromaDB)
     ↓
Top-k relevant documents (context)
     ↓
Prompt = context + question
     ↓
Language model generates grounded answer (GPT-2)
```

**Note on GPT-2 outputs:** GPT-2 is a small (~117M param) model with no instruction tuning — its outputs are creative but not always factually grounded. In production RAG pipelines, instruction-tuned models (Mistral, LLaMA-3, GPT-4) produce much more coherent Q&A responses.